# Data Integration — HMIS Core Join
**MedTrack_DV — Milestone 1, Step: Build `hospital_overview_dataset.csv`**

HMIS is the approved CORE dataset. This notebook:
1. Loads the key HMIS tables (`admission`, `patient`, `department`, `ward`, `bed`, `disease`)
2. Joins them at the **admission grain** (one row = one admission) — exactly as the mentor's document specifies for Hospital Overview
3. Prints the structure of `staff_assignment` and `ward` separately, so the next notebook (Resource Utilization + Department Analytics) can be planned with real column names

This notebook does NOT touch Beds Management / Readmission / Inpatient Discharges — those remain standalone supplementary sources per the mentor's instruction ("do not force unrelated datasets together").

In [1]:
import pandas as pd
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

HMIS_DIR = "../data/raw/hmis"
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

## 1. Load core HMIS tables

In [2]:
admission = pd.read_csv(f"{HMIS_DIR}/admission.csv")
patient = pd.read_csv(f"{HMIS_DIR}/patient.csv")
department = pd.read_csv(f"{HMIS_DIR}/department.csv")
ward = pd.read_csv(f"{HMIS_DIR}/ward.csv")
bed = pd.read_csv(f"{HMIS_DIR}/bed.csv")
disease = pd.read_csv(f"{HMIS_DIR}/disease.csv")

print("admission :", admission.shape, list(admission.columns))
print("patient   :", patient.shape, list(patient.columns))
print("department:", department.shape, list(department.columns))
print("ward      :", ward.shape, list(ward.columns))
print("bed       :", bed.shape, list(bed.columns))
print("disease   :", disease.shape, list(disease.columns))

admission : (45000, 10) ['admission_id', 'admission_date', 'discharge_date', 'admission_type', 'admission_status', 'patient_id', 'department_id', 'ward_id', 'bed_id', 'disease_id']
patient   : (30000, 6) ['patient_id', 'gender', 'date_of_birth', 'blood_group', 'city', 'contact_number']
department: (11, 5) ['department_id', 'department_name', 'department_type', 'floor_number', 'status']
ward      : (27, 5) ['ward_id', 'ward_name', 'ward_type', 'total_beds', 'department_id']
bed       : (415, 4) ['bed_id', 'bed_number', 'bed_status', 'ward_id']
disease   : (20, 3) ['disease_id', 'disease_name', 'disease_category']


## 2. Validate join keys BEFORE merging
Per mentor's rule: check for unmatched IDs before joining, so we know exactly how many admissions will be dropped/kept.

In [3]:
def check_keys(left, right, key, left_name, right_name):
    left_ids = set(left[key].dropna().unique())
    right_ids = set(right[key].dropna().unique())
    missing_in_right = left_ids - right_ids
    print(f"{left_name}.{key} not found in {right_name}: {len(missing_in_right)} / {len(left_ids)}")

check_keys(admission, patient, 'patient_id', 'admission', 'patient')
check_keys(admission, department, 'department_id', 'admission', 'department')
check_keys(admission, ward, 'ward_id', 'admission', 'ward')
check_keys(admission, bed, 'bed_id', 'admission', 'bed')
check_keys(admission, disease, 'disease_id', 'admission', 'disease')

admission.patient_id not found in patient: 0 / 23275
admission.department_id not found in department: 0 / 6
admission.ward_id not found in ward: 0 / 27
admission.bed_id not found in bed: 0 / 145
admission.disease_id not found in disease: 0 / 20


## 3. Build `hospital_overview_dataset` — grain: one row = one admission
Left join from `admission` (the fact table) so every admission is preserved even if a lookup ID is somehow unmatched (would show as NaN, flagged afterward).

In [4]:
hospital_overview = (
    admission
    .merge(patient, on='patient_id', how='left', suffixes=('', '_patient'))
    .merge(department, on='department_id', how='left', suffixes=('', '_dept'))
    .merge(ward, on='ward_id', how='left', suffixes=('', '_ward'))
    .merge(bed, on='bed_id', how='left', suffixes=('', '_bed'))
    .merge(disease, on='disease_id', how='left', suffixes=('', '_disease'))
)

print("Joined shape:", hospital_overview.shape)
print("Original admission rows:", admission.shape[0])
assert hospital_overview.shape[0] == admission.shape[0], "Row count changed — check for duplicate keys in lookup tables!"
hospital_overview.head()

Joined shape: (45000, 28)
Original admission rows: 45000


,admission_id,admission_date,discharge_date,admission_type,admission_status,patient_id,department_id,ward_id,bed_id,disease_id,gender,date_of_birth,blood_group,city,contact_number,department_name,department_type,floor_number,status,ward_name,ward_type,total_beds,department_id_ward,bed_number,bed_status,ward_id_bed,disease_name,disease_category
0,1,2020-02-25,2020-02-27,Emergency,Discharged,166,2,6,76,10,Male,1954-11-02,A-,North Dorisland,2486990142,Internal Medicine,Clinical,2,Active,Internal Medicine Ward 1,Private,10,2,6-1,Available,6,Anemia,Hematological
1,2,2022-02-22,2022-03-04,Elective,Discharged,8622,5,21,302,11,Female,1988-02-25,AB+,West Susan,301.488.2792x8582,Orthopedics,Clinical,4,Active,Orthopedics Ward 1,General,15,5,21-2,Available,21,Fracture Femur,Orthopedic
2,3,2021-02-03,2021-02-09,Elective,Discharged,23976,1,2,11,9,Male,1944-03-02,A-,North Erik,562.729.3026,Emergency,Clinical,0,Active,Emergency Ward 2,Private,15,1,2-1,Available,2,Chronic Obstructive Pulmonary Disease,Respiratory
3,4,2021-12-31,2022-01-05,Elective,Discharged,16635,2,10,128,1,Male,2017-08-19,A+,Maldonadoton,329.219.6005x16344,Internal Medicine,Clinical,2,Active,Internal Medicine Ward 5,General,20,2,10-8,Available,10,Acute Myocardial Infarction,Cardiac
4,5,2022-07-02,2022-07-07,Elective,Discharged,10654,3,11,157,7,Male,2015-05-06,A+,Port Stephanie,001-589-318-6174,Surgery,Clinical,3,Active,Surgery Ward 1,General,20,3,11-17,Available,11,Hypertension,Cardiac


## 4. Parse dates & derive Length of Stay

In [5]:
hospital_overview['admission_date'] = pd.to_datetime(hospital_overview['admission_date'], errors='coerce', dayfirst=True)
hospital_overview['discharge_date'] = pd.to_datetime(hospital_overview['discharge_date'], errors='coerce', dayfirst=True)

print("admission_date parse failures:", hospital_overview['admission_date'].isna().sum())
print("discharge_date parse failures:", hospital_overview['discharge_date'].isna().sum())

hospital_overview['length_of_stay_days'] = (
    hospital_overview['discharge_date'] - hospital_overview['admission_date']
).dt.days

print("\nLOS summary:")
print(hospital_overview['length_of_stay_days'].describe())

admission_date parse failures: 0
discharge_date parse failures: 0

LOS summary:
count    45000.000000
mean         5.155000
std          3.271284
min          1.000000
25%          3.000000
50%          4.000000
75%          8.000000
max         15.000000
Name: length_of_stay_days, dtype: float64


C:\Users\memyo\AppData\Local\Temp\ipykernel_35564\1802103273.py:1: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  hospital_overview['admission_date'] = pd.to_datetime(hospital_overview['admission_date'], errors='coerce', dayfirst=True)
C:\Users\memyo\AppData\Local\Temp\ipykernel_35564\1802103273.py:2: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  hospital_overview['discharge_date'] = pd.to_datetime(hospital_overview['discharge_date'], errors='coerce', dayfirst=True)


## 5. Outlier validation (per mentor's rule)
Check for impossible values before deciding to clean/exclude.

In [6]:
print("Negative LOS rows:", (hospital_overview['length_of_stay_days'] < 0).sum())
print("LOS > 365 days rows:", (hospital_overview['length_of_stay_days'] > 365).sum())
if 'age' in hospital_overview.columns:
    print("Age < 0:", (hospital_overview['age'] < 0).sum())
    print("Age > 120:", (hospital_overview['age'] > 120).sum())

Negative LOS rows: 0
LOS > 365 days rows: 0


## 6. Derive Readmission Flag (documented formula)
**Definition used:** a patient is a readmission if they have another admission for the SAME patient_id within 30 days of a previous discharge_date.
**Denominator:** all admissions where the patient had at least one prior discharge (i.e., not their first-ever admission).
This is derived directly from HMIS `admission` — not cross-matched with the external Readmission dataset, since patient IDs are not shared across datasets (documented limitation).

In [7]:
ho_sorted = hospital_overview.sort_values(['patient_id', 'admission_date']).copy()
ho_sorted['prev_discharge_date'] = ho_sorted.groupby('patient_id')['discharge_date'].shift(1)
ho_sorted['days_since_prev_discharge'] = (ho_sorted['admission_date'] - ho_sorted['prev_discharge_date']).dt.days

ho_sorted['is_readmission'] = (
    ho_sorted['days_since_prev_discharge'].notna() &
    (ho_sorted['days_since_prev_discharge'] <= 30) &
    (ho_sorted['days_since_prev_discharge'] >= 0)
)

hospital_overview = ho_sorted.drop(columns=['prev_discharge_date', 'days_since_prev_discharge'])

eligible = hospital_overview['patient_id'].duplicated(keep=False).sum()
readmit_count = hospital_overview['is_readmission'].sum()
print(f"Readmissions flagged: {readmit_count}")
print(f"Readmission rate (of all admissions): {readmit_count / hospital_overview.shape[0] * 100:.2f}%")

Readmissions flagged: 986
Readmission rate (of all admissions): 2.19%


## 7. Save `hospital_overview_dataset.csv`

In [8]:
hospital_overview.to_csv(f"{PROCESSED_DIR}/hospital_overview_dataset.csv", index=False)
print("Saved:", f"{PROCESSED_DIR}/hospital_overview_dataset.csv")
print("Final shape:", hospital_overview.shape)

Saved: ../data/processed/hospital_overview_dataset.csv
Final shape: (45000, 30)


## 8. Inspect remaining tables — for planning Department Analytics & Resource Utilization
Print structure only. Do NOT build these tables yet — confirm columns first.

In [9]:
staff_assignment = pd.read_csv(f"{HMIS_DIR}/staff_assignment.csv")
employee = pd.read_csv(f"{HMIS_DIR}/employee.csv")

print("staff_assignment:", staff_assignment.shape, list(staff_assignment.columns))
print(staff_assignment.head())
print()
print("employee:", employee.shape, list(employee.columns))
print(employee.head())

staff_assignment: (207, 4) ['assignment_id', 'employee_id', 'ward_id', 'shift']
   assignment_id  employee_id  ward_id    shift
0              1            2       24  Morning
1              2            4        2  Morning
2              3           17       24    Night
3              4           19       12    Night
4              5           20       22  Morning

employee: (500, 7) ['employee_id', 'employee_name', 'gender', 'role', 'employment_type', 'date_of_joining', 'department_id']
   employee_id  employee_name  gender        role employment_type date_of_joining  department_id
0            1   Sanaya Kalla    Male       Admin        Contract      2024-04-17             11
1            2    Dayamai Raj  Female       Nurse       Full-time      2011-05-09              6
2            3     Logan Lata    Male      Doctor        Contract      2020-10-27              8
3            4  Vyanjana Kota  Female  Technician        Contract      2013-05-01              9
4            5  Nachi

In [10]:
print("ward columns:", list(ward.columns))
print(ward.head())
print()
print("bed columns:", list(bed.columns))
print(bed.head())
print()
print("department columns:", list(department.columns))
print(department.head())

ward columns: ['ward_id', 'ward_name', 'ward_type', 'total_beds', 'department_id']
   ward_id         ward_name ward_type  total_beds  department_id
0        1  Emergency Ward 1   General          10              1
1        2  Emergency Ward 2   Private          15              1
2        3  Emergency Ward 3   General          10              1
3        4  Emergency Ward 4   General          20              1
4        5  Emergency Ward 5   General          20              1

bed columns: ['bed_id', 'bed_number', 'bed_status', 'ward_id']
   bed_id bed_number bed_status  ward_id
0       1        1-1   Occupied        1
1       2        1-2   Occupied        1
2       3        1-3  Available        1
3       4        1-4   Occupied        1
4       5        1-5   Occupied        1

department columns: ['department_id', 'department_name', 'department_type', 'floor_number', 'status']
   department_id    department_name department_type  floor_number  status
0              1          Emergenc

## Next steps (planning only — do not build yet)
1. Review `staff_assignment` / `employee` / `ward` / `bed` / `department` columns printed above.
2. Determine whether `bed` and `staff_assignment` have any date/time field — if NOT, Resource Utilization and Department Analytics cannot be truly "per day"; they will instead be a **snapshot** (current capacity/allocation), and daily trends will be derived from `hospital_overview` (admissions/discharges per day, occupied beds per day computed from admission/discharge date ranges).
3. Once confirmed, build:
   - `patient_flow_dataset.csv` from `hospital_overview` (admission + discharge events; HMIS has no granular ward-transfer log, so this is documented as a limitation — flow is admission-to-discharge, not multi-step ward movement)
   - `department_analytics_dataset.csv` — group `hospital_overview` by `department_id` + admission day
   - `resource_utilization_dataset.csv` — combine `bed`, `ward`, `staff_assignment` with daily occupancy computed from `hospital_overview`'s admission/discharge date ranges